# Dynamic Programming: Optimal Bike Fleet Deployment

**Assignment:** IT-5082 Optimization Methods — Programming Assignment  
**Student:** Priyanka PDMK (MS25952070)  
**Method:** Exact Method — Dynamic Programming (DP)  
**Dataset:** UCI Bike Sharing Dataset 

---

## Overview

**Dynamic Programming** solution to the **Optimal Hourly Bike Fleet Deployment** problem.

Given a fixed fleet of bikes and hourly demand data, we determine the optimal number of bikes to make **available** at each hour of the day to **maximize total demand served**, subject to:
- Fleet size constraint (total bikes is fixed)
- Non-negativity (cannot deploy negative bikes)
- Return rate (bikes used are partially returned each hour)

### Why I'm using Dynamic Programming?
This problem has two key properties that make DP applicable:
1. **Optimal Substructure**: The optimal deployment from hour `t` to `T` depends only on the state at hour `t` (bikes available), not on how we got there.
2. **Overlapping Subproblems**: Many deployment scenarios share the same (hour, bikes_available) state DP avoids recomputing these.

DP gives us the **guaranteed globally optimal solution** unlike heuristics, it never misses the true optimum.

---
## 1. Setup & Data Loading

In [9]:
%pip install ucimlrepo --quiet


[notice] A new release of pip is available: 26.0 -> 26.0.1
[notice] To update, run: pip3 install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [6]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns
import time
from functools import lru_cache

plt.style.use('seaborn-v0_8-darkgrid')
sns.set_palette('husl')
print("Libraries loaded successfully.")

Libraries loaded successfully.


In [8]:
# ── Load dataset from local CSV (UCI Bike Sharing Dataset) ───────────────────
from pathlib import Path


def _find_hour_csv() -> Path:
    """Find hour.csv whether cwd is repo root, notebooks/, or notebooks/MS25952070/."""
    here = Path.cwd().resolve()
    candidates = [
        here / "data" / "bike+sharing+dataset" / "hour.csv",
        here.parent / "data" / "bike+sharing+dataset" / "hour.csv",
        here.parent.parent / "data" / "bike+sharing+dataset" / "hour.csv",
        here / "data" / "hour.csv",
        here.parent / "data" / "hour.csv",
    ]
    for p in candidates:
        if p.is_file():
            return p
    tried = "\n".join(f"  - {p}" for p in candidates)
    raise FileNotFoundError(
        "Could not find hour.csv. Clone the repo so data/bike+sharing+dataset/hour.csv exists, or adjust cwd. Tried:\n"
        + tried
    )


csv_path = _find_hour_csv()
hour_data = pd.read_csv(csv_path)

print(f"Loaded from: {csv_path}")
print(f"Dataset loaded: {hour_data.shape[0]:,} hourly records")
print(f"Date range: {hour_data['dteday'].min()} → {hour_data['dteday'].max()}")
print(f"Columns: {list(hour_data.columns)}")

Loaded from: /Users/UKALAMO/Desktop/Assign/IT-5082-OptimizationMethods/data/bike+sharing+dataset/hour.csv
Dataset loaded: 17,379 hourly records
Date range: 2011-01-01 → 2012-12-31
Columns: ['instant', 'dteday', 'season', 'yr', 'mnth', 'hr', 'holiday', 'weekday', 'workingday', 'weathersit', 'temp', 'atemp', 'hum', 'windspeed', 'casual', 'registered', 'cnt']


---
## 2. Demand Profile Extraction

I extract the **average hourly demand** across all days as our demand vector `d[h]` for `h ∈ {0, 1, ..., 23}`.  
This captures the characteristic daily demand pattern used to drive the DP.

In [10]:
# ── Aggregate demand by hour of day ───────────────────────────────────────────
hourly_demand = hour_data.groupby("hr")["cnt"].mean().values  # shape: (24,)
hourly_demand_max = hour_data.groupby("hr")["cnt"].max().values
hourly_demand_std = hour_data.groupby("hr")["cnt"].std().values

# Global stats (from analysis.ipynb)
avg_demand  = hour_data["cnt"].mean()   # ≈ 189.46 bikes/hr
std_demand  = hour_data["cnt"].std()    # ≈ 181.39
max_demand  = hour_data["cnt"].max()    # 977
fleet_size  = int(avg_demand + 2 * std_demand)  # 95% confidence fleet

print(f"Average hourly demand : {avg_demand:.1f} bikes/hr")
print(f"Std dev               : {std_demand:.1f} bikes/hr")
print(f"Peak single hour      : {max_demand} bikes")
print(f"Derived fleet size    : {fleet_size} bikes  (μ + 2σ, 95% confidence)")
print(f"\nHourly avg demand profile:")
for h, d in enumerate(hourly_demand):
    bar = '█' * int(d / 15)
    print(f"  {h:02d}:00  {d:6.1f}  {bar}")

Average hourly demand : 189.5 bikes/hr
Std dev               : 181.4 bikes/hr
Peak single hour      : 977 bikes
Derived fleet size    : 552 bikes  (μ + 2σ, 95% confidence)

Hourly avg demand profile:
  00:00    53.9  ███
  01:00    33.4  ██
  02:00    22.9  █
  03:00    11.7  
  04:00     6.4  
  05:00    19.9  █
  06:00    76.0  █████
  07:00   212.1  ██████████████
  08:00   359.0  ███████████████████████
  09:00   219.3  ██████████████
  10:00   173.7  ███████████
  11:00   208.1  █████████████
  12:00   253.3  ████████████████
  13:00   253.7  ████████████████
  14:00   240.9  ████████████████
  15:00   251.2  ████████████████
  16:00   312.0  ████████████████████
  17:00   461.5  ██████████████████████████████
  18:00   425.5  ████████████████████████████
  19:00   311.5  ████████████████████
  20:00   226.0  ███████████████
  21:00   172.3  ███████████
  22:00   131.3  ████████
  23:00    87.8  █████
